# Extract Labels từ Facts

Notebook này đọc toàn bộ `facts/P*.jsonl` (output sau khi filter của `build_kg.py`) và tách nhãn ra thành hai file độc lập:

| File output | Nội dung |
|---|---|
| `facts/qid_labels.json` | `{ QID → label }` — nhãn của mọi entity xuất hiện trong facts |
| `facts/pid_labels.json` | `{ PID → label }` — nhãn của mọi relation xuất hiện trong facts |

> **Tại sao tách riêng?**  
> `cache/labels.json` và `cache/relation_labels.json` chứa toàn bộ dump (rất lớn).  
> Hai file mới chỉ chứa những QID/PID thực sự dùng trong bộ facts đã lọc — nhỏ hơn nhiều và dễ load hơn khi generate câu hỏi.

## 0. Imports & config

In [1]:
import json
from pathlib import Path

try:
    import orjson
    _loads = orjson.loads
    _dumps = lambda obj: orjson.dumps(obj, option=orjson.OPT_INDENT_2).decode()
    print("dùng orjson (nhanh hơn)")
except ImportError:
    _loads = json.loads
    _dumps = lambda obj: json.dumps(obj, ensure_ascii=False, indent=2)
    print("dùng json stdlib")

FACTS_DIR = Path("facts")
assert FACTS_DIR.exists(), f"{FACTS_DIR} chưa tồn tại — chạy build_kg.py trước"

jsonl_files = sorted(FACTS_DIR.glob("P*.jsonl"))
print(f"Tìm thấy {len(jsonl_files)} file P*.jsonl trong {FACTS_DIR}/")

dùng orjson (nhanh hơn)
Tìm thấy 271 file P*.jsonl trong facts/


## 1. Đọc facts và thu thập labels

Mỗi dòng trong `P*.jsonl` có dạng:
```json
{"s_qid": "Q123", "s_label": "...", "o_qid": "Q456", "o_label": "...",
 "start": 2010, "end": 2015, "relation": "P39", "r_label": "giữ chức"}
```

Ta tách `(s_qid, s_label)` và `(o_qid, o_label)` vào `qid_labels`,  
và `(relation, r_label)` vào `pid_labels`.

In [2]:
from collections import Counter
from tqdm.auto import tqdm

qid_labels: dict[str, dict] = {}  # QID -> {"label": ..., "lang": ...}
pid_labels: dict[str, dict] = {}  # PID -> {"label": ..., "lang": ...}
lang_counts: dict[str, Counter] = {
    "s_lang": Counter(),
    "o_lang": Counter(),
    "r_lang": Counter(),
}
total_facts = 0
skipped = 0

for fpath in tqdm(jsonl_files, desc="đọc file"):
    with fpath.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                fact = _loads(line)
            except Exception:
                skipped += 1
                continue

            total_facts += 1

            if fact.get("s_qid") and fact.get("s_label"):
                qid_labels[fact["s_qid"]] = {"label": fact["s_label"], "lang": fact.get("s_lang")}
            if fact.get("o_qid") and fact.get("o_label"):
                qid_labels[fact["o_qid"]] = {"label": fact["o_label"], "lang": fact.get("o_lang")}
            if fact.get("relation") and fact.get("r_label"):
                pid_labels[fact["relation"]] = {"label": fact["r_label"], "lang": fact.get("r_lang")}

            for k in ("s_lang", "o_lang", "r_lang"):
                lang_counts[k][fact.get(k) or "(missing)"] += 1

print(f"\nTổng facts đọc : {total_facts:,}")
print(f"Dòng lỗi bỏ qua: {skipped:,}")
print(f"QID unique      : {len(qid_labels):,}")
print(f"PID unique      : {len(pid_labels):,}")

đọc file:   0%|          | 0/271 [00:00<?, ?it/s]


Tổng facts đọc : 237,907
Dòng lỗi bỏ qua: 0
QID unique      : 37,584
PID unique      : 271


## 2. Kiểm tra nhanh

In [3]:
from collections import Counter

qid_lang_counts = Counter(info['lang'] or '(missing)' for info in qid_labels.values())
pid_lang_counts = Counter(info['lang'] or '(missing)' for info in pid_labels.values())

print(f"QID unique: {len(qid_labels):,}")
for lang, n in qid_lang_counts.most_common():
    print(f"  {lang:<12} {n:>8,} ({100*n/len(qid_labels):.2f}%)")

print(f"\nPID unique: {len(pid_labels):,}")
for lang, n in pid_lang_counts.most_common():
    print(f"  {lang:<12} {n:>8,} ({100*n/len(pid_labels):.2f}%)")


QID unique: 37,584
  en             25,064 (66.69%)
  vi             12,520 (33.31%)

PID unique: 271
  vi                208 (76.75%)
  en                 63 (23.25%)


## 3. Ghi ra file

Lưu hai file JSON vào thư mục `facts/`:
- `qid_labels.json` — map từ QID sang nhãn (ưu tiên tiếng Việt nếu có)
- `pid_labels.json` — map từ PID sang nhãn relation

In [4]:
qid_out = FACTS_DIR / "qid_labels.json"
pid_out = FACTS_DIR / "pid_labels.json"

qid_out.write_text(_dumps(qid_labels), encoding="utf-8")
pid_out.write_text(_dumps(pid_labels), encoding="utf-8")

print(f"Đã ghi: {qid_out}  ({qid_out.stat().st_size / 1024:.1f} KB)")
print(f"Đã ghi: {pid_out}  ({pid_out.stat().st_size / 1024:.1f} KB)")


Đã ghi: facts\qid_labels.json  (2979.1 KB)
Đã ghi: facts\pid_labels.json  (20.0 KB)


## 4. Thống kê cuối

Phân bố coverage label theo từng file P*.jsonl.

In [5]:
print(f"{'PID':<8} {'relation label':<35} {'#facts':>8}")
print("-" * 55)

for fpath in sorted(jsonl_files):
    pid = fpath.stem
    count = sum(1 for _ in fpath.open(encoding="utf-8"))
    info = pid_labels.get(pid)
    label = f"{info['label']} [{info['lang']}]" if info else "(no label)"
    print(f"{pid:<8} {label:<35} {count:>8,}")


PID      relation label                        #facts
-------------------------------------------------------
P1000    record held [en]                           8
P1001    thuộc quyền tài phán [vi]                187
P101     lĩnh vực làm việc [vi]                    20
P102     đảng viên của đảng chính trị [vi]      1,121
P1027    được trao bởi [vi]                        34
P1028    donated by [en]                            7
P1029    (các) người lái [vi]                     166
P1037    người quản lý/giám đốc [vi]              120
P1038    họ hàng [vi]                               2
P1050    tình trạng sức khỏe [vi]                 119
P1056    sản xuất [vi]                              3
P106     nghề nghiệp [vi]                       1,852
P10611   chứng nhận [vi]                            3
P1064    khổ đường ray [vi]                        11
P1066    học trò của [vi]                           5
P1071    vị trí sáng tạo [vi]                      23
P1075    hiệu trưởng [vi] 

## 5. Dịch label EN → VI bằng Ollama

Những QID/PID có `lang != "vi"` (chủ yếu là `en` hoặc `(missing)`) sẽ được dịch sang tiếng Việt qua Ollama HTTP API.  
Đổi `OLLAMA_URL` / `OLLAMA_MODEL` cho khớp với máy đang chạy Ollama.

In [ ]:
import os, json, time, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

OLLAMA_URL   = os.environ.get("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen3.5:4b")  # sửa lại tên đúng
TIMEOUT      = 120
WORKERS      = 4  # khớp với OLLAMA_NUM_PARALLEL

# giả định FACTS_DIR đã định nghĩa ở trên
QID_TRANSLATED = FACTS_DIR / "qid_labels.vi.json"
PID_TRANSLATED = FACTS_DIR / "pid_labels.vi.json"

SYSTEM_PROMPT = (
    "Bạn là chuyên gia dịch thuật dữ liệu. "
    "Dịch label/relation sau từ tiếng Anh sang tiếng Việt một cách chuẩn chỉnh, tự nhiên, ngắn gọn, giữ nguyên tên riêng. "
    "Chỉ trả về bản dịch, KHÔNG giải thích, KHÔNG dấu ngoặc kép, KHÔNG tiền tố."
)

session = requests.Session()  # reuse TCP connection, nhanh hơn

def translate(text: str) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": text,
        "system": SYSTEM_PROMPT,
        "stream": False,
        "think": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0.0,
            "num_ctx": 1024,       # label ngắn, 512 đủ
            "num_predict": 128,     # label ngắn, 128 đủ
        },
    }
    r = session.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()["response"].strip().strip('"').strip()

def translate_batch(texts, workers=WORKERS, show_progress=True):
    """Dịch nhiều text song song. Trả về dict {text: translation}."""
    results = {}
    total = len(texts)
    start = time.time()

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(translate, t): t for t in texts}
        for i, f in enumerate(as_completed(futures), 1):
            t = futures[f]
            try:
                results[t] = f.result()
            except Exception as e:
                print(f"  FAIL: {t!r} -> {e}")
                results[t] = None
            if show_progress and i % 20 == 0:
                elapsed = time.time() - start
                rate = i / elapsed
                eta = (total - i) / rate
                print(f"  [{i}/{total}] {rate:.1f} item/s, ETA {eta:.0f}s")
    return results

def translate_with_cache(texts, cache_path: Path, workers=WORKERS, save_every=100):
    """Dịch + cache ra JSON, resume được nếu crash giữa chừng."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}

    todo = [t for t in texts if t not in cache]
    print(f"Cache hit: {len(cache)} | Cần dịch: {len(todo)} / {len(texts)}")

    if not todo:
        return cache

    done_since_save = 0
    start = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(translate, t): t for t in todo}
        for i, f in enumerate(as_completed(futures), 1):
            t = futures[f]
            try:
                cache[t] = f.result()
            except Exception as e:
                print(f"  FAIL {t!r}: {e}")
                continue

            done_since_save += 1
            if done_since_save >= save_every:
                cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
                done_since_save = 0
                rate = i / (time.time() - start)
                print(f"  [{i}/{len(todo)}] saved | {rate:.1f} item/s")

    cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Done in {time.time()-start:.1f}s")
    return cache

# ---------- smoke test ----------
print("Ollama OK:", translate("head of government"))

# test batch nhỏ
sample = ["country", "human", "located in", "instance of", "date of birth"]
print(translate_batch(sample))

Ollama OK: Chủ tịch chính phủ
{'country': 'Quốc gia', 'date of birth': 'Ngày sinh', 'human': 'người', 'located in': 'nằm tại', 'instance of': 'thể hiện của'}


In [18]:
TESTS = [
    # relations
    "head of government",
    "member of political party",
    "educated at",
    "position held",
    "award received",
    "country of citizenship",
    # entities — tên riêng phải giữ nguyên
    "Barack Obama",
    "University of Oxford",
    "Marathon world record progression",
    # khái niệm
    "prime minister",
    "Nobel Prize in Physics",
    "academic thesis",
]

for src in TESTS:
    try:
        vi = translate(src)
    except Exception as e:
        vi = f"<lỗi: {e}>"
    print(f"  {src:<40} → {vi}")


  head of government                       → Chủ tịch chính phủ
  member of political party                → Thành viên của đảng chính trị
  educated at                              → Đã được đào tạo tại
  position held                            → Chức vụ đảm nhiệm
  award received                           → Giải thưởng đã nhận
  country of citizenship                   → Quốc tịch
  Barack Obama                             → Barack Obama
  University of Oxford                     → Đại học Oxford
  Marathon world record progression        → Tiến trình kỷ lục thế giới chạy marathon
  prime minister                           → Thủ tướng
  Nobel Prize in Physics                   → Giải Nobel Vật lý
  academic thesis                          → luận văn khoa học


### 5.1 Lọc các label cần dịch

In [14]:
qid_todo = {qid: info for qid, info in qid_labels.items() if info.get("lang") != "vi"}
pid_todo = {pid: info for pid, info in pid_labels.items() if info.get("lang") != "vi"}

print(f"QID cần dịch: {len(qid_todo):,} / {len(qid_labels):,}")
print(f"PID cần dịch: {len(pid_todo):,} / {len(pid_labels):,}")

QID cần dịch: 25,064 / 37,584
PID cần dịch: 63 / 271


### 5.2 Chạy dịch (resume-safe)

Kết quả được merge vào `qid_labels.vi.json` / `pid_labels.vi.json`. Chạy lại sẽ chỉ dịch những item còn thiếu.

In [39]:
import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# orjson nhanh hơn nếu có, không thì dùng json
try:
    import orjson
    def _dumps(obj): return orjson.dumps(obj, option=orjson.OPT_INDENT_2 | orjson.OPT_NON_STR_KEYS).decode()
    def _loads(s):   return orjson.loads(s)
except ImportError:
    def _dumps(obj): return json.dumps(obj, ensure_ascii=False, indent=2)
    def _loads(s):   return json.loads(s)

WORKERS = 4  # khớp với OLLAMA_NUM_PARALLEL

def load_cache(path):
    if path.exists():
        return _loads(path.read_text(encoding="utf-8"))
    return {}

def save_cache(path, data):
    path.write_text(_dumps(data), encoding="utf-8")

def run_translate(todo: dict, out_path, kind: str,
                  save_every: int = 50, workers: int = WORKERS):
    cache = load_cache(out_path)
    pending = [(k, v) for k, v in todo.items() if k not in cache]
    print(f"[{kind}] đã dịch: {len(cache):,} | còn lại: {len(pending):,}")

    if not pending:
        return cache

    failed = 0
    done_since_save = 0

    with ThreadPoolExecutor(max_workers=workers) as ex:
        # submit toàn bộ job
        future_to_item = {
            ex.submit(translate, info["label"]): (key, info)
            for key, info in pending
        }

        pbar = tqdm(as_completed(future_to_item),
                    total=len(pending),
                    desc=f"dịch {kind}")

        for fut in pbar:
            key, info = future_to_item[fut]
            try:
                vi = fut.result()
            except Exception as e:
                failed += 1
                if failed <= 5:
                    tqdm.write(f"  lỗi {key} ({info['label']!r}): {e}")
                continue

            cache[key] = {
                "label": vi,
                "lang": "vi",
                "src_label": info["label"],
                "src_lang": info.get("lang"),
            }

            done_since_save += 1
            if done_since_save >= save_every:
                save_cache(out_path, cache)
                done_since_save = 0
                pbar.set_postfix(saved=len(cache), failed=failed)

    save_cache(out_path, cache)
    print(f"[{kind}] xong. Tổng cache: {len(cache):,} | lỗi: {failed:,}")
    return cache


# === Chạy ===
pid_vi = run_translate(pid_todo, PID_TRANSLATED, "PID")  # ít, chạy trước
qid_vi = run_translate(qid_todo, QID_TRANSLATED, "QID")

# === Flag để review & dump Excel ===
import re, pandas as pd

VN_DIACRITICS = re.compile(r"[ăâđêôơưĂÂĐÊÔƠƯáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵÁÀẢÃẠẤẦẨẪẬẮẰẲẴẶÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ]")
BAD_TOKENS = ("bản dịch", "translation:", "dịch là", "sang tiếng", "\n")

def flag_row(src_label: str, src_lang, vi_label: str) -> str:
    if not vi_label:
        return "empty"
    low = vi_label.lower()
    if any(t in low for t in BAD_TOKENS) or any(c in vi_label for c in '"(){}[]:'):
        return "bad_format"
    if len(vi_label) > max(20, 3 * len(src_label)):
        return "too_long"
    if vi_label.strip() == src_label.strip():
        return "unchanged"
    has_vn = bool(VN_DIACRITICS.search(vi_label))
    # nếu src là tiếng Anh nhiều chữ mà bản dịch không có dấu VN nào → nghi ngờ
    if not has_vn and len(vi_label.split()) >= 2 and re.fullmatch(r"[A-Za-z0-9 .,'\-/&]+", vi_label):
        return "no_diacritics"
    return "ok"

def to_df(cache: dict, key_name: str) -> pd.DataFrame:
    rows = []
    for k, v in cache.items():
        src_label = v.get("src_label") or ""
        vi_label  = v.get("label") or ""
        rows.append({
            key_name:   k,
            "src_label": src_label,
            "src_lang":  v.get("src_lang"),
            "vi_label":  vi_label,
            "flag":      flag_row(src_label, v.get("src_lang"), vi_label),
        })
    df = pd.DataFrame(rows)
    # sort: flag != ok lên đầu, rồi theo key
    df["_p"] = (df["flag"] == "ok").astype(int)
    return df.sort_values(["_p", "flag", key_name]).drop(columns="_p").reset_index(drop=True)

pid_df = to_df(pid_vi, "PID")
qid_df = to_df(qid_vi, "QID")

print("PID flag breakdown:")
print(pid_df["flag"].value_counts().to_string())
print("\nQID flag breakdown:")
print(qid_df["flag"].value_counts().to_string())

xlsx_path = FACTS_DIR / "translations.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
    pid_df.to_excel(w, sheet_name="PID", index=False)
    qid_df.to_excel(w, sheet_name="QID", index=False)

print(f"\nĐã ghi Excel: {xlsx_path}  ({xlsx_path.stat().st_size / 1024:.1f} KB)")


[PID] đã dịch: 63 | còn lại: 0
[QID] đã dịch: 25,064 | còn lại: 0
PID flag breakdown:
flag
ok            62
bad_format     1

QID flag breakdown:
flag
ok               15808
unchanged         8438
bad_format         432
no_diacritics      363
too_long            23

Đã ghi Excel: facts/translations.xlsx  (1107.8 KB)


### 5.3 Merge bản dịch vào `qid_labels` / `pid_labels`

Ghi đè file `facts/qid_labels.json` và `facts/pid_labels.json` với label đã dịch sang VI (giữ `src_label` / `src_lang` để tra ngược).

In [ ]:
qid_merged = dict(qid_labels)
for qid, vi_info in qid_vi.items():
    qid_merged[qid] = vi_info

pid_merged = dict(pid_labels)
for pid, vi_info in pid_vi.items():
    pid_merged[pid] = vi_info

(FACTS_DIR / "qid_labels.json").write_text(_dumps(qid_merged), encoding="utf-8")
(FACTS_DIR / "pid_labels.json").write_text(_dumps(pid_merged), encoding="utf-8")

from collections import Counter
print("QID lang sau merge:", Counter(v['lang'] or '(missing)' for v in qid_merged.values()))
print("PID lang sau merge:", Counter(v['lang'] or '(missing)' for v in pid_merged.values()))